In [22]:
import pandas as pd

# Carregar o dataset
df = pd.read_csv('players_data-2024_2025.csv')

# 1. Limpar linhas de cabeçalho repetidas (comum no FBref)
df = df[df['Player'] != 'Player']

# 2. Converter colunas essenciais para numérico (trata erros de strings)
cols_to_fix = ['Min', 'Gls', 'Ast', 'xG', 'PrgP', 'Tkl']
for col in cols_to_fix:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# 3. Criar um filtro de "Minutos Mínimos"
# Isto evita que um jogador que jogou 2 minutos e marcou 1 golo tenha stats infinitas
df_main = df[df['Min'] > 450].copy() 

# 4. Criar métricas "Per 90" (Ajuste justo para quem joga mais/menos)
df_main['Gls_90'] = (df_main['Gls'] / df_main['Min']) * 90
df_main['SCA_90'] = (pd.to_numeric(df['SCA90'], errors='coerce')) # Já tens esta coluna pronta!

# 5. Ver o top 5 criadores de jogo (SCA90)
print("--- TOP CRIADORES (Shot Creating Actions per 90) ---")
df_main[['Player', 'Squad', 'SCA90']].sort_values(by='SCA90', ascending=False).head(5)

--- TOP CRIADORES (Shot Creating Actions per 90) ---


,Player,Squad,SCA90
2525,Suso,Sevilla,6.68
1960,Michael Olise,Bayern Munich,6.51
1221,Isco,Betis,6.46
541,Rayan Cherki,Lyon,6.44
2381,Léo Scienza,Heidenheim,6.39


In [23]:
import pandas as pd

df = pd.read_csv('players_data-2024_2025.csv')

print(df.columns.tolist())

['Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'xG', 'npxG', 'xAG', 'npxG+xAG', 'PrgC', 'PrgP', 'PrgR', 'G+A-PK', 'xG+xAG', 'Rk_stats_shooting', 'Nation_stats_shooting', 'Pos_stats_shooting', 'Comp_stats_shooting', 'Age_stats_shooting', 'Born_stats_shooting', '90s_stats_shooting', 'Gls_stats_shooting', 'Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'Dist', 'FK', 'PK_stats_shooting', 'PKatt_stats_shooting', 'xG_stats_shooting', 'npxG_stats_shooting', 'npxG/Sh', 'G-xG', 'np:G-xG', 'Rk_stats_passing', 'Nation_stats_passing', 'Pos_stats_passing', 'Comp_stats_passing', 'Age_stats_passing', 'Born_stats_passing', '90s_stats_passing', 'Cmp', 'Att', 'Cmp%', 'TotDist', 'PrgDist', 'Ast_stats_passing', 'xAG_stats_passing', 'xA', 'A-xAG', 'KP', '1/3', 'PPA', 'CrsPA', 'PrgP_stats_passing', 'Rk_stats_passing_types', 'Nation_stats_passing_types', 'Pos_stats_passing_types', 'Comp_stats

In [24]:
import pandas as pd

# 1. Carregar o original
df = pd.read_csv('players_data-2024_2025.csv')

# 2. Definir a nossa "Lista de Desejos" atualizada
colunas_finais = [
    # Bio & Estatísticas de Equipa (Impacto no jogo)
    'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Min', '90s',
    'onG', 'onGA', '+/-', '+/-90', 'onxG', 'onxGA',
    
    # Ataque (Eficiência e Volume)
    'Gls', 'Sh', 'SoT', 'xG', 'npxG', 'G/Sh', 'G-xG', 'Dist',
    
    # Passe (Construção e Ruptura)
    'Ast', 'Cmp%', 'KP', 'PPA', 'CrsPA', 'PrgP', 'TB',
    
    # Criação de Perigo (Drible e Provocação)
    'SCA90', 'GCA90', 'TO', 'Fld',
    
    # Defesa (Combativividade e Posicionamento)
    'Tkl', 'Int', 'Blocks', 'Clr', 'Tkl%',
    
    # Posse (Controlo e Progressão)
    'Touches', 'Succ', 'Succ%', 'PrgC',
    
    # Outros (Disciplina e Duelos)
    'CrdY', 'CrdR', 'Recov', 'Won%',
    
    # Keeper (Métricas Específicas)
    'GA', 'Saves', 'Save%', 'PSxG', 'CS', 'Launch%'
]

# 3. Filtrar e limpar
# Nota: O FBref por vezes tem linhas de cabeçalho no meio do CSV, removemos aqui:
df = df[df['Player'] != 'Player']

# Criamos o master apenas com as colunas que escolhemos
df_master = df[colunas_finais].copy()

# 4. Tratamento de Dados (Converter para números e filtrar minutos)
# Garantimos que as colunas numéricas são tratadas como tal
for col in colunas_finais:
    if col not in ['Player', 'Nation', 'Pos', 'Squad', 'Comp']:
        df_master[col] = pd.to_numeric(df_master[col], errors='coerce').fillna(0)

# Filtro de 270 min (3 jogos completos) para dar fiabilidade aos dados
df_master = df_master[df_master['Min'] >= 270]

# 5. Guardar o novo ficheiro
df_master.to_csv('players_master.csv', index=False)
print(f"Sucesso! O teu dataset agora tem {df_master.shape[1]} colunas e {df_master.shape[0]} jogadores.")

Sucesso! O teu dataset agora tem 52 colunas e 2210 jogadores.


In [25]:
import sklearn
print(f"Sucesso! Versão do sklearn: {sklearn.__version__}")

Sucesso! Versão do sklearn: 1.7.2


In [26]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# 1. Carregar os dados
df = pd.read_csv('players_master.csv')

# 2. Preparar o Motor (Scaling e Similaridade)
def prepare_engine(dataframe):
    # Separamos o que é texto do que é número
    cat_cols = ['Player', 'Nation', 'Pos', 'Squad', 'Comp']
    num_cols = [col for col in dataframe.columns if col not in cat_cols]
    
    # Normalizamos 
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(dataframe[num_cols])
    
    # Calculamos a matriz de similaridade 
    sim_matrix = cosine_similarity(scaled_data)
    
    return sim_matrix

# 3. Função de Recomendação
def get_recommendations(player_name, dataframe, sim_matrix, top_n=5):
    if player_name not in dataframe['Player'].values:
        return f"Jogador '{player_name}' não encontrado."
    
    # Encontra o índice do jogador no CSV
    idx = dataframe[dataframe['Player'] == player_name].index[0]
    
    sim_scores = list(enumerate(sim_matrix[idx]))
    # Ordena do mais parecido para o menos parecido
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    top_indices = [i[0] for i in sim_scores[1:top_n+1]]

    return dataframe.iloc[top_indices][['Player', 'Squad', 'Pos', 'Comp']]

def get_recommendations_v2(player_name, dataframe, sim_matrix, 
                             target_pos=None, 
                             max_age=None, 
                             comp=None, 
                             top_n=5):
    
    if player_name not in dataframe['Player'].values:
        return f"Jogador '{player_name}' não encontrado."
    
    # 1. Encontrar o índice do jogador base
    idx = dataframe[dataframe['Player'] == player_name].index[0]
    
    # 2. Obter os scores de similaridade
    sim_scores = list(enumerate(sim_matrix[idx]))
    
    # 3. Criar um DataFrame temporário com os scores para podermos filtrar
    # Criamos uma cópia do original e adicionamos a coluna 'Similarity'
    df_temp = dataframe.copy()
    df_temp['Similarity'] = [score[1] for score in sim_scores]
    
    # 4. APLICAR FILTROS
    # Filtrar por Posição (ex: 'DF')
    if target_pos:
        # Usamos .str.contains porque o FBref às vezes tem 'DF,MF'
        df_temp = df_temp[df_temp['Pos'].str.contains(target_pos, na=False)]
    
    # Filtrar por Idade Máxima
    if max_age:
        df_temp = df_temp[df_temp['Age'] <= max_age]
        
    # Filtrar por Liga (Comp)
    if comp:
        df_temp = df_temp[df_temp['Comp'].str.contains(comp, na=False)]

    # 5. Ordenar e retornar (excluindo o próprio jogador)
    # Removemos o jogador que pesquisámos da lista de resultados
    df_temp = df_temp[df_temp['Player'] != player_name]
    
    # Ordenamos pela similaridade e pegamos no Top N
    final_results = df_temp.sort_values(by='Similarity', ascending=False).head(top_n)
    
    return final_results[['Player', 'Squad', 'Pos', 'Age', 'Comp', 'Similarity']]

def prepare_weighted_engine(dataframe, weights=None):
    # 1. Separar colunas
    cat_cols = ['Player', 'Nation', 'Pos', 'Squad', 'Comp']
    df_numeric = dataframe.drop(columns=cat_cols).fillna(0)
    
    # 2. Normalizar (StandardScaler)
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_numeric)
    
    # Criamos um DataFrame para manipular as colunas escalonadas
    df_scaled = pd.DataFrame(scaled_data, columns=df_numeric.columns)
    
    # 3. APLICAR PESOS
    # Se passarmos pesos, multiplicamos as colunas específicas
    # Exemplo de pesos: {'Gls': 2.0, 'Ast': 1.5}
    if weights:
        for col, weight in weights.items():
            if col in df_scaled.columns:
                df_scaled[col] = df_scaled[col] * weight
    
    # 4. Calcular Similaridade com os novos valores pesados
    return cosine_similarity(df_scaled.values)

# --- TESTE DE CENÁRIOS ---

# Cenário 1: "Olheiro de Goleadores" 
# Queremos dar muita importância a Golos, Remates e xG
pesos_avancado = {
    'Gls': 3.0, 
    'xG': 2.0, 
    'Sh': 1.5,
    'Tkl': 0.1, # Ignoramos quase totalmente a defesa
    'Int': 0.1
}

matrix_avancados = prepare_weighted_engine(df, weights=pesos_avancado)

search = "Erling Haaland"
print(f"--- 🎯 Resultados Pesados para: {search} (Foco em Finalização) ---")
print(get_recommendations_v2(search, df, matrix_avancados, top_n=5))

--- 🎯 Resultados Pesados para: Erling Haaland (Foco em Finalização) ---
                    Player           Squad Pos   Age                Comp  \
1142    Robert Lewandowski       Barcelona  FW  35.0          es La Liga   
512      Ermedin Demirović       Stuttgart  FW  26.0       de Bundesliga   
815        Serhou Guirassy        Dortmund  FW  28.0       de Bundesliga   
1285  Jean-Philippe Mateta  Crystal Palace  FW  27.0  eng Premier League   
580           Artem Dovbyk            Roma  FW  27.0          it Serie A   

      Similarity  
1142    0.981737  
512     0.979374  
815     0.974853  
1285    0.974272  
580     0.973651  


In [28]:
import numpy as np

def calculate_market_index(dataframe):
    df_idx = dataframe.copy()
    
    # 1. Pontuação de Idade (Inversa: quanto mais novo, melhor)
    # Jogadores com 18-21 anos recebem pontuação máxima
    df_idx['Age_Score'] = df_idx['Age'].apply(lambda x: 100 if x <= 21 else max(0, 100 - (x - 21) * 7))
    
    # 2. Pontuação de Performance (Normalizada de 0 a 100)
    # Vamos usar uma mistura de G+A, SCA90 e Recov para uma nota geral
    perf_metrics = ['Gls', 'Ast', 'SCA90', 'Recov']
    for col in perf_metrics:
        # Normalizamos de 0 a 100
        df_idx[f'{col}_Score'] = (df_idx[col] - df_idx[col].min()) / (df_idx[col].max() - df_idx[col].min()) * 100
    
    df_idx['Performance_Score'] = df_idx[[f'{c}_Score' for c in perf_metrics]].mean(axis=1)
    
    # 3. CÁLCULO FINAL: O "Market Index"
    # Pesos: 40% Idade, 60% Performance
    df_idx['Market_Index'] = (df_idx['Age_Score'] * 0.4) + (df_idx['Performance_Score'] * 0.6)
    
    # Arredondar para ser mais bonito no site
    df_idx['Market_Index'] = df_idx['Market_Index'].round(1)
    
    return df_idx

# Atualizar o nosso DataFrame
df = calculate_market_index(df)

# Guardar de novo no CSV para o Backend já ter estes dados
df.to_csv('players_master.csv', index=False)

# --- TESTAR O ÍNDICE ---
print("--- 💎 TOP 'PECHINCHAS' (Alto Índice de Mercado) ---")
# Filtramos por jogadores que não são super estrelas (ex: fora da Premier League ou idade baixa)
df.sort_values(by='Market_Index', ascending=False)[['Player', 'Age', 'Squad', 'Market_Index']].head(10)

--- 💎 TOP 'PECHINCHAS' (Alto Índice de Mercado) ---


,Player,Age,Squad,Market_Index
1508,Michael Olise,22.0,Bayern Munich,74.6
2162,Lamine Yamal,17.0,Barcelona,74.2
2149,Florian Wirtz,21.0,Leverkusen,72.8
1580,Pedri,21.0,Barcelona,71.4
394,Rayan Cherki,20.0,Lyon,70.7
1557,Cole Palmer,22.0,Chelsea,68.7
170,Bradley Barcola,21.0,Paris S-G,67.3
1865,Xavi Simons,21.0,RB Leipzig,66.4
1577,Nicolás Paz,19.0,Como,66.1
140,Alex Baena,23.0,Villarreal,65.2
